# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [9]:
df.head()

,table,definition
0,employees,"emp_id, name, age, department"


In [10]:
df.columns

Index(['table', 'definition'], dtype='object')

In [8]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': ['employees'],
        'definition': ['emp_id, name, age, department']}
df = pd.DataFrame(data)
print(df)

       table                     definition
0  employees  emp_id, name, age, department


In [11]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [12]:
print(text_tables)

employees: emp_id, name, age, department


In [13]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [15]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="List the names of employees who work in the IT department")

In [16]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees"]
}
```


In [18]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="How many employees are in each department?")

In [19]:
print(return_OAI(pqt3))

```json
{
    "tables": ["employees"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

1. Setting up the Schema

In [ ]:
import pandas as pd

# “catalog” of tables used by the prompts
df = pd.DataFrame([
    {"table": "employees", "definition": "emp_id, name, age, department"},
    {"table": "salaries",  "definition": "emp_id, salary, bonus"},
    {"table": "studies",   "definition": "emp_id, degree, field"}
])


text_tables = "\n".join(f"{row.table}: {row.definition}" for _, row in df.iterrows())
text_tables

'employees: emp_id, name, age, department\nsalaries: emp_id, salary, bonus\nstudies: emp_id, degree, field'

In [24]:
prompt_sql_from_question = """
You are a careful SQL generator.

SCHEMA
{tables}

INSTRUCTIONS
- Use only the tables/columns shown above.
- Assume ANSI SQL (works on SQLite/Postgres).
- Do not invent columns or tables.
- Return ONLY a fenced SQL code block. No explanations.

QUESTION
{question}
"""

2. Prompts

In [25]:
#Question 1
p_sql = prompt_sql_from_question.format(
    tables=text_tables,
    question="List the names and ages of employees in the IT department"
)
print(return_OAI(p_sql))

```sql
SELECT name, age
FROM employees
WHERE department = 'IT';
```


In [32]:
#Question 2
pqt = prompt_question_tables.format(tables=text_tables, question="List employees and their job titles")
print(return_OAI(pqt))

{
    "tables": ["employees"]
}


In [33]:
print(return_OAI(prompt_sql_from_question.format(
    tables=text_tables,
    question="List employees and their job titles"
)))

```sql
SELECT e.emp_id, e.name, e.department
FROM employees e;
```


In [ ]:
#Question 3
pqt = prompt_question_tables.format(tables=text_tables, question="Show the name and age of employees older than the company average age")
print(return_OAI(pqt))

{
    "tables": ["employees"]
}


In [27]:
print(return_OAI(prompt_sql_from_question.format(
    tables=text_tables,
    question="Show the name and age of employees older than the company average age"
)))

```sql
SELECT name, age
FROM employees
WHERE age > (SELECT AVG(age) FROM employees);
```


In [34]:
#Question 4
pqt = prompt_question_tables.format(tables=text_tables, question="Show all the departments and their costs")
print(return_OAI(pqt))

```json
{
    "tables": ["employees", "salaries"]
}
```


In [35]:
print(return_OAI(prompt_sql_from_question.format(
    tables=text_tables,
    question="Show all the departments and their costs"
)))

```sql
SELECT e.department, SUM(s.salary + s.bonus) AS total_cost
FROM employees e
JOIN salaries s ON e.emp_id = s.emp_id
GROUP BY e.department;
```


In [ ]:
#Question 5
pqt = prompt_question_tables.format(tables=text_tables, question="How many employees have a Master's or PhD, grouped by department?")
print(return_OAI(pqt))

{
    "tables": ["employees", "studies"]
}


In [31]:
print(return_OAI(prompt_sql_from_question.format(
    tables=text_tables,
    question="How many employees have a Master's or PhD, grouped by department?"
)))

```sql
SELECT department, COUNT(emp_id)
FROM studies
WHERE degree IN ('Master\'s', 'PhD')
GROUP BY department;
```


# One‑Page Report — Natural Language Questions into SQL

**Objective** Convert natural-language questions into SQL using small schema (exployees, salaries, studies). Evaluate prompt variations and identify failure models. 

## Method
1. Build a schema catalog as a DataFrame and rendered it to a text bloc (text_tables). 
2. Used the notebook´s prompt_question_tables template with {tables} and {questions} to ask multiple questions. 
3. Chedked wheteher the assistant selected the correct tables and produced sensible SQL. 

## Findings
- Clear, column-aligned questions. The model reliably selected employees only and generated correct SQL.
- Robustness imporved when prompts contained exact columns names from the schemas.

## What did not work well
- Hallucinations in some of the queries "cost" table was not found, and SQL gave as result "employees" not accurate. Or in question #2 "job titles" as well. 

## Recommendations & learnings 
- Always to include the schema text in the prompt and use exact column names. 
- For joins, name both tables in the question e.g: "Using the employees and salaries tables, list the top 3 highest-paid employees (name and salary)."
- Small wording can impact the table selection. 
